In [ ]:
import csv
import itertools
import random
import math
from collections import defaultdict, Counter
import pandas as pd

# Создаем 10 случайных наборов виньеток, максимально отличных друг от друга

# Сначала по 9 штук в каждой

# Потом в каждом наборе уберем по две виньетки, оставляя в каждом виньетку 9 (это контрольная группа). Далее нужно убирать виньетки так, чтобы итоговое количество комбинаций было как можно более разнообразным



In [ ]:
SEED = 42
N_SETS = 10
OUTPUT = "vignettes_10_sets_after_removal.xlsx"

CONTROL_VIGNETTE_ID = 9
REMOVE_PER_SET = 2
FINAL_VIGNETTES_PER_SET = 7

random.seed(SEED)

GENDERS = ["мужчина", "женщина"]
AGES = ["30 лет", "46 лет", "68 лет"]

VIGNETTES_POLICY = [
    {
        "id": 1,
        "policy_A": ("Кандидат А поддерживает действующую власть, а также "
                     "выступает за обязательное проведение уроков патриотического воспитания в школах."),
        "policy_B": ("Кандидат Б не поддерживает действующую власть и высказывается против обязательного проведения "
                     "уроков патриотического воспитания в школах."),
    },
    {
        "id": 2,
        "policy_A": ("Кандидат А поддерживает действующую власть и высказывается против обязательного проведения "
                     "уроков патриотического воспитания в школах."),
        "policy_B": ("Кандидат Б не поддерживает действующую власть и выступает за обязательное проведение уроков патриотического воспитания в школах."),
    },
    {
        "id": 3,
        "policy_A": ("Кандидат А поддерживает действующую власть и выступает за обязательное проведение уроков патриотического воспитания в школах."),
        "policy_B": ("Кандидат Б не поддерживает действующую власть, а также выступает за обязательное проведение уроков патриотического воспитания в школах."),
    },
    {
        "id": 4,
        "policy_A": ("Кандидат А поддерживает действующую власть, а также высказывается против обязательного проведения "
                     "уроков патриотического воспитания в школах."),
        "policy_B": ("Кандидат Б не поддерживает действующую власть и при этом высказывается против обязательного проведения "
                     "уроков патриотического воспитания в школах."),
    },
    {
        "id": 5,
        "policy_A": ("Кандидат А поддерживает действующую власть и выступает "
                     "в поддержку легализации эвтаназии."),
        "policy_B": ("Кандидат Б не поддерживает действующую власть и выступает против "
                     "легализации эвтаназии."),
    },
    {
        "id": 6,
        "policy_A": ("Кандидат А поддерживает действующую власть и выступает против "
                     "легализации эвтаназии."),
        "policy_B": ("Кандидат Б не поддерживает действующую власть и выступает "
                     "в поддержку легализации эвтаназии."),
    },
    {
        "id": 7,
        "policy_A": ("Кандидат А поддерживает действующую власть, а также выступает "
                     "в поддержку легализации эвтаназии."),
        "policy_B": ("Кандидат Б не поддерживает действующую власть и выступает "
                     "в поддержку легализации эвтаназии."),
    },
    {
        "id": 8,
        "policy_A": ("Кандидат А поддерживает действующую власть и выступает против "
                     "легализации эвтаназии."),
        "policy_B": ("Кандидат Б не поддерживает действующую власть и выступает против "
                     "легализации эвтаназии."),
    },
    {
        "id": 9,
        "policy_A": "Кандидат А поддерживает действующую власть.",
        "policy_B": "Кандидат Б не поддерживает действующую власть.",
    },
]



def build_pool():
    pool = {}

    for vignette in VIGNETTES_POLICY:
        vid = vignette["id"]
        combos = []
        idx = 0

        for ag, aa, bg, ba in itertools.product(GENDERS, AGES, GENDERS, AGES):
            idx += 1

            if ag == bg and aa == ba:
                continue

            text_A = f"Кандидат А — {ag} {aa}. {vignette['policy_A']}"
            text_B = f"Кандидат Б — {bg} {ba}. {vignette['policy_B']}"

            combos.append({
                "vignette_id": vid,
                "combo_id": f"{vid}_{idx:03d}",
                "A_gender": ag,
                "A_age": aa,
                "B_gender": bg,
                "B_age": ba,
                "text_A": text_A,
                "text_B": text_B,
                "full_vignette": f"{text_A}\n\n{text_B}",
            })

        pool[vid] = combos

    return pool


def generate_full_random_sets(pool, n_sets):


    all_rows = []
    vignette_ids = sorted(pool.keys())

    for set_id in range(1, n_sets + 1):
        for position_in_set, vid in enumerate(vignette_ids, start=1):
            chosen = random.choice(pool[vid])

            row = chosen.copy()
            row["set_id"] = set_id
            row["original_position_in_set"] = position_in_set

            all_rows.append(row)

    return pd.DataFrame(all_rows)


def profile_key(row):


    return (
        row["A_gender"],
        row["A_age"],
        row["B_gender"],
        row["B_age"],
    )


def choose_rows_to_remove_from_ready_sets(
    df,
    control_vignette_id=9,
    remove_per_set=2,
    seed=42,
):

    rng = random.Random(seed)

    df = df.copy()

    # Считаем, какие профили пола/возраста чаще встречаются в полном датасете
    df["_profile"] = df.apply(profile_key, axis=1)
    profile_counts = Counter(df["_profile"])

    removable_vignette_ids = sorted(
        vid for vid in df["vignette_id"].unique()
        if vid != control_vignette_id
    )

    possible_drop_pairs = list(
        itertools.combinations(removable_vignette_ids, remove_per_set)
    )

    removal_counts = Counter()
    used_drop_pairs = set()
    rows_to_drop = []
    removal_plan = {}

    for set_id in sorted(df["set_id"].unique()):
        set_df = df[df["set_id"] == set_id]

        candidate_pairs = possible_drop_pairs[:]
        rng.shuffle(candidate_pairs)

        best_pair = None
        best_score = None

        for pair in candidate_pairs:
            pair = tuple(sorted(pair))

            rows_pair = set_df[set_df["vignette_id"].isin(pair)]

            if len(rows_pair) != remove_per_set:
                continue


            profile_redundancy = sum(
                profile_counts[p] for p in rows_pair["_profile"]
            )

            score = (
                pair in used_drop_pairs,

                sum(removal_counts[vid] for vid in pair),
                max(removal_counts[vid] for vid in pair),

                -profile_redundancy,
            )

            if best_score is None or score < best_score:
                best_score = score
                best_pair = pair

        if best_pair is None:
            raise ValueError(f"Не удалось выбрать виньетки для удаления в наборе {set_id}")

        removal_plan[set_id] = best_pair
        used_drop_pairs.add(best_pair)

        for vid in best_pair:
            removal_counts[vid] += 1

        drop_index = set_df[set_df["vignette_id"].isin(best_pair)].index.tolist()
        rows_to_drop.extend(drop_index)

        for idx in drop_index:
            profile_counts[df.loc[idx, "_profile"]] -= 1

    df_final = df.drop(index=rows_to_drop).copy()

    df_final = df_final.drop(columns=["_profile"])

    df_final = df_final.sort_values(["set_id", "vignette_id"]).copy()
    df_final["position_in_set"] = (
        df_final.groupby("set_id").cumcount() + 1
    )

    df_final["removed_vignettes"] = df_final["set_id"].map(
        lambda x: ",".join(map(str, removal_plan[x]))
    )

    return df_final, removal_plan, removal_counts


def validate_output(
    df,
    expected_n_sets,
    final_vignettes_per_set,
    control_vignette_id,
):
    invalid = df[
        (df["A_gender"] == df["B_gender"]) &
        (df["A_age"] == df["B_age"])
    ]

    if len(invalid) > 0:
        raise ValueError(
            f"Найдены строки, где кандидаты полностью совпадают: {len(invalid)}"
        )

    counts_by_set = df.groupby("set_id")["vignette_id"].nunique()

    bad_sets = counts_by_set[counts_by_set != final_vignettes_per_set]

    if len(bad_sets) > 0:
        raise ValueError(
            f"Есть наборы, где не {final_vignettes_per_set} уникальных виньеток:\n{bad_sets}"
        )

    control_counts = (
        df[df["vignette_id"] == control_vignette_id]
        .groupby("set_id")
        .size()
    )

    bad_control_sets = [
        set_id for set_id in range(1, expected_n_sets + 1)
        if control_counts.get(set_id, 0) != 1
    ]

    if bad_control_sets:
        raise ValueError(
            f"В этих наборах нет ровно одной контрольной виньетки "
            f"{control_vignette_id}: {bad_control_sets}"
        )

    duplicates = df.duplicated(subset=["set_id", "vignette_id"], keep=False)

    if duplicates.any():
        raise ValueError(
            "Есть повторяющиеся vignette_id внутри одного set_id."
        )

    print("Проверки пройдены:")
    print(f"- в каждом наборе {final_vignettes_per_set} виньеток")
    print(f"- виньетка {control_vignette_id} сохранена в каждом наборе")
    print("- кандидаты внутри виньетки не совпадают полностью")
    print("- внутри набора нет повторов vignette_id")


def main():
    pool = build_pool()

    print("Валидных комбинаций на виньетку:")
    for vid, combos in pool.items():
        print(f"Виньетка {vid}: {len(combos)}")

    df_full = generate_full_random_sets(
        pool=pool,
        n_sets=N_SETS,
    )

    print(f"\nСначала создано строк: {len(df_full)}")
    print(f"Это {N_SETS} наборов × 9 виньеток")

    df_final, removal_plan, removal_counts = choose_rows_to_remove_from_ready_sets(
        df=df_full,
        control_vignette_id=CONTROL_VIGNETTE_ID,
        remove_per_set=REMOVE_PER_SET,
        seed=SEED,
    )

    cols = [
        "set_id",
        "position_in_set",
        "original_position_in_set",
        "vignette_id",
        "combo_id",
        "A_gender",
        "A_age",
        "B_gender",
        "B_age",
        "removed_vignettes",
        "text_A",
        "text_B",
        "full_vignette",
    ]

    df_final = df_final[cols]

    validate_output(
        df=df_final,
        expected_n_sets=N_SETS,
        final_vignettes_per_set=FINAL_VIGNETTES_PER_SET,
        control_vignette_id=CONTROL_VIGNETTE_ID,
    )

    print("\nПлан удаления:")
    for set_id, removed in removal_plan.items():
        print(f"Набор {set_id}: удалены виньетки {removed}")

    print("\nЧастоты удаления виньеток:")
    for vid in sorted(removal_counts):
        print(f"Виньетка {vid}: удалена {removal_counts[vid]} раз")

    df_final.to_excel(OUTPUT, index=False)

    print(f"\nСохранено {len(df_final)} строк → {OUTPUT}")


if __name__ == "__main__":
    main()

Валидных комбинаций на виньетку:
Виньетка 1: 30
Виньетка 2: 30
Виньетка 3: 30
Виньетка 4: 30
Виньетка 5: 30
Виньетка 6: 30
Виньетка 7: 30
Виньетка 8: 30
Виньетка 9: 30

Сначала создано строк: 90
Это 10 наборов × 9 виньеток
Проверки пройдены:
- в каждом наборе 7 виньеток
- виньетка 9 сохранена в каждом наборе
- кандидаты внутри виньетки не совпадают полностью
- внутри набора нет повторов vignette_id

План удаления:
Набор 1: удалены виньетки (np.int64(2), np.int64(6))
Набор 2: удалены виньетки (np.int64(1), np.int64(5))
Набор 3: удалены виньетки (np.int64(4), np.int64(8))
Набор 4: удалены виньетки (np.int64(3), np.int64(7))
Набор 5: удалены виньетки (np.int64(2), np.int64(3))
Набор 6: удалены виньетки (np.int64(6), np.int64(7))
Набор 7: удалены виньетки (np.int64(5), np.int64(8))
Набор 8: удалены виньетки (np.int64(1), np.int64(4))
Набор 9: удалены виньетки (np.int64(3), np.int64(8))
Набор 10: удалены виньетки (np.int64(2), np.int64(5))

Частоты удаления виньеток:
Виньетка 1: удалена 2 р

In [ ]:

import re

INPUT_FILE = "vignettes_10_sets_after_removal.xlsx"
OUTPUT_FILE = "vignettes_10_sets_after_removal_with_codes.xlsx"


VIGNETTE_CODE_MAP = {
    1: "Pfa",
    2: "Paf",
    3: "Pff",
    4: "Paa",
    5: "Efa",
    6: "Eaf",
    7: "Eff",
    8: "Eaa",
    9: "Nopos",
}


def gender_code(value):
    value = str(value).strip().lower()

    if value == "мужчина":
        return "M"
    if value == "женщина":
        return "F"

    raise ValueError(f"Неизвестное значение пола: {value}")


def age_code(value):
    """
    Превращает:
    '30 лет' -> '30'
    '46 лет' -> '46'
    '68 лет' -> '68'
    """
    value = str(value).strip()
    match = re.search(r"\d+", value)

    if not match:
        raise ValueError(f"Не удалось извлечь возраст из значения: {value}")

    return match.group(0)


def build_vignette_code(row):
    base_code = VIGNETTE_CODE_MAP.get(int(row["vignette_id"]))

    if base_code is None:
        raise ValueError(f"Неизвестный vignette_id: {row['vignette_id']}")

    candidate_a_code = (
        gender_code(row["A_gender"])
        + age_code(row["A_age"])
    )

    candidate_b_code = (
        gender_code(row["B_gender"])
        + age_code(row["B_age"])
    )

    return base_code + candidate_a_code + candidate_b_code


def main():
    df = pd.read_excel(INPUT_FILE)

    required_columns = [
        "vignette_id",
        "A_gender",
        "A_age",
        "B_gender",
        "B_age",
    ]

    missing_columns = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"В файле не хватает столбцов: {missing_columns}"
        )

    df["vignette_code"] = df.apply(build_vignette_code, axis=1)

    df.to_excel(OUTPUT_FILE, index=False)

    print(f"Готово: {OUTPUT_FILE}")
    print(df[[
        "vignette_id",
        "A_gender",
        "A_age",
        "B_gender",
        "B_age",
        "vignette_code",
    ]].head(10))


if __name__ == "__main__":
    main()

Готово: vignettes_10_sets_after_removal_with_codes.xlsx
   vignette_id A_gender   A_age B_gender   B_age vignette_code
0            1  женщина  46 лет  мужчина  30 лет     PfaF46M30
1            3  мужчина  30 лет  мужчина  46 лет     PffM30M46
2            4  женщина  46 лет  женщина  30 лет     PaaF46F30
3            5  мужчина  46 лет  женщина  46 лет     EfaM46F46
4            7  мужчина  46 лет  женщина  30 лет     EffM46F30
5            8  мужчина  30 лет  женщина  68 лет     EaaM30F68
6            9  женщина  46 лет  женщина  30 лет   NoposF46F30
7            2  женщина  46 лет  мужчина  46 лет     PafF46M46
8            3  женщина  46 лет  женщина  30 лет     PffF46F30
9            4  женщина  68 лет  женщина  30 лет     PaaF68F30


# 3.Теперь внутри каждого набора нужно сделать случайный порядок показа виньеток

In [ ]:
SEED = 42

INPUT_FILE = "vignettes_10_sets_after_removal_with_codes.xlsx"
OUTPUT_FILE = "vignettes_10_sets_after_removal_with_codes_shuffled.xlsx"


def shuffle_vignettes_within_sets(df, seed=42):
    required_columns = ["set_id", "vignette_id"]

    missing_columns = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing_columns:
        raise ValueError(f"В файле не хватает столбцов: {missing_columns}")

    df_shuffled = (
        df
        .groupby("set_id", group_keys=False)
        .apply(lambda x: x.sample(frac=1, random_state=seed + int(x.name)))
        .reset_index(drop=True)
    )

    df_shuffled["position_in_set"] = (
        df_shuffled
        .groupby("set_id")
        .cumcount() + 1
    )

    return df_shuffled


def validate_output(df):
    counts_by_set = df.groupby("set_id")["vignette_id"].nunique()

    bad_sets = counts_by_set[counts_by_set != 7]

    if len(bad_sets) > 0:
        raise ValueError(
            f"Есть наборы, где не 7 уникальных виньеток:\n{bad_sets}"
        )

    control_counts = (
        df[df["vignette_id"] == 9]
        .groupby("set_id")
        .size()
    )

    bad_control_sets = [
        set_id for set_id in df["set_id"].unique()
        if control_counts.get(set_id, 0) != 1
    ]

    if bad_control_sets:
        raise ValueError(
            f"В этих наборах нет ровно одной виньетки 9: {bad_control_sets}"
        )

    bad_positions = []

    for set_id, group in df.groupby("set_id"):
        positions = sorted(group["position_in_set"].tolist())

        if positions != list(range(1, 8)):
            bad_positions.append(set_id)

    if bad_positions:
        raise ValueError(
            f"Некорректные position_in_set в наборах: {bad_positions}"
        )

    print("Проверки пройдены:")
    print("- в каждом наборе 7 виньеток")
    print("- виньетка 9 сохранена в каждом наборе")
    print("- position_in_set пересчитана от 1 до 7")


def main():
    df = pd.read_excel(INPUT_FILE)

    df_shuffled = shuffle_vignettes_within_sets(
        df=df,
        seed=SEED,
    )

    validate_output(df_shuffled)

    df_shuffled.to_excel(OUTPUT_FILE, index=False)

    print(f"Сохранено: {OUTPUT_FILE}")


if __name__ == "__main__":
    main()

Проверки пройдены:
- в каждом наборе 7 виньеток
- виньетка 9 сохранена в каждом наборе
- position_in_set пересчитана от 1 до 7
Сохранено: vignettes_10_sets_after_removal_with_codes_shuffled.xlsx


/tmp/ipykernel_45226/4105719980.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(frac=1, random_state=seed + int(x.name)))


In [ ]:
import re
import pandas as pd
from scipy.stats import chi2_contingency


INPUT_FILE = "vignettes_10_sets_after_removal_with_codes_shuffled.xlsx"
OUTPUT_FILE = "vignettes_balance_chi_square_tests.xlsx"


def extract_base_code(vignette_code):
    """
    Примеры:
    PfaM30F46 -> Pfa
    EafF68M30 -> Eaf
    NoposM30F46 -> Nopos
    """
    value = str(vignette_code)

    if value.startswith("Nopos"):
        return "Nopos"

    return value[:3]


def run_chi_square(table, test_name):
    """
    Запускает χ²-тест и возвращает результаты.
    """

    chi2, p_value, dof, expected = chi2_contingency(table)

    min_expected = expected.min()
    share_expected_lt_5 = (expected < 5).sum() / expected.size

    result = {
        "test": test_name,
        "chi2": chi2,
        "p_value": p_value,
        "dof": dof,
        "min_expected_count": min_expected,
        "share_expected_cells_lt_5": share_expected_lt_5,
        "n_rows": table.shape[0],
        "n_cols": table.shape[1],
    }

    return result, pd.DataFrame(
        expected,
        index=table.index,
        columns=table.columns,
    )


def safe_sheet_name(name):
    """
    Excel не любит длинные названия листов.
    """
    name = re.sub(r"[\[\]\:\*\?\/\\]", "_", name)
    return name[:31]


def main():
    df = pd.read_excel(INPUT_FILE)

    required_columns = [
        "set_id",
        "vignette_id",
        "vignette_code",
        "A_gender",
        "A_age",
        "B_gender",
        "B_age",
    ]

    missing_columns = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing_columns:
        raise ValueError(f"В файле не хватает столбцов: {missing_columns}")

    # Базовый код экспериментального условия
    df["base_code"] = df["vignette_code"].apply(extract_base_code)

    # Тип темы: P = high/patriotic, E = low/euthanasia, Nopos = контроль
    df["issue_block"] = df["base_code"].apply(
        lambda x: "P" if x.startswith("P")
        else "E" if x.startswith("E")
        else "Nopos"
    )

    tests = []
    tables = {}
    expected_tables = {}

    # ------------------------------------------------------------
    # 1. Отличается ли распределение пола у кандидата A и B?
    # ------------------------------------------------------------
    table = pd.DataFrame({
        "A_candidate": df["A_gender"].value_counts(),
        "B_candidate": df["B_gender"].value_counts(),
    }).fillna(0).astype(int)

    result, expected = run_chi_square(
        table,
        "A_vs_B_gender_distribution",
    )

    tests.append(result)
    tables["A_vs_B_gender"] = table
    expected_tables["A_vs_B_gender_expected"] = expected

    # ------------------------------------------------------------
    # 2. Отличается ли распределение возраста у кандидата A и B?
    # ------------------------------------------------------------
    table = pd.DataFrame({
        "A_candidate": df["A_age"].value_counts(),
        "B_candidate": df["B_age"].value_counts(),
    }).fillna(0).astype(int)

    result, expected = run_chi_square(
        table,
        "A_vs_B_age_distribution",
    )

    tests.append(result)
    tables["A_vs_B_age"] = table
    expected_tables["A_vs_B_age_expected"] = expected

    # ------------------------------------------------------------
    # 3. Баланс пола A-кандидата по vignette_id
    # ------------------------------------------------------------
    table = pd.crosstab(df["vignette_id"], df["A_gender"])

    result, expected = run_chi_square(
        table,
        "A_gender_by_vignette_id",
    )

    tests.append(result)
    tables["A_gender_by_vid"] = table
    expected_tables["A_gender_by_vid_expected"] = expected

    # ------------------------------------------------------------
    # 4. Баланс пола B-кандидата по vignette_id
    # ------------------------------------------------------------
    table = pd.crosstab(df["vignette_id"], df["B_gender"])

    result, expected = run_chi_square(
        table,
        "B_gender_by_vignette_id",
    )

    tests.append(result)
    tables["B_gender_by_vid"] = table
    expected_tables["B_gender_by_vid_expected"] = expected

    # ------------------------------------------------------------
    # 5. Баланс возраста A-кандидата по vignette_id
    # ------------------------------------------------------------
    table = pd.crosstab(df["vignette_id"], df["A_age"])

    result, expected = run_chi_square(
        table,
        "A_age_by_vignette_id",
    )

    tests.append(result)
    tables["A_age_by_vid"] = table
    expected_tables["A_age_by_vid_expected"] = expected

    # ------------------------------------------------------------
    # 6. Баланс возраста B-кандидата по vignette_id
    # ------------------------------------------------------------
    table = pd.crosstab(df["vignette_id"], df["B_age"])

    result, expected = run_chi_square(
        table,
        "B_age_by_vignette_id",
    )

    tests.append(result)
    tables["B_age_by_vid"] = table
    expected_tables["B_age_by_vid_expected"] = expected

    # ------------------------------------------------------------
    # 7. Баланс пола A-кандидата по base_code: Pfa/Paf/etc.
    # ------------------------------------------------------------
    table = pd.crosstab(df["base_code"], df["A_gender"])

    result, expected = run_chi_square(
        table,
        "A_gender_by_base_code",
    )

    tests.append(result)
    tables["A_gender_by_code"] = table
    expected_tables["A_gender_by_code_expected"] = expected

    # ------------------------------------------------------------
    # 8. Баланс пола B-кандидата по base_code
    # ------------------------------------------------------------
    table = pd.crosstab(df["base_code"], df["B_gender"])

    result, expected = run_chi_square(
        table,
        "B_gender_by_base_code",
    )

    tests.append(result)
    tables["B_gender_by_code"] = table
    expected_tables["B_gender_by_code_expected"] = expected

    # ------------------------------------------------------------
    # 9. Баланс возраста A-кандидата по base_code
    # ------------------------------------------------------------
    table = pd.crosstab(df["base_code"], df["A_age"])

    result, expected = run_chi_square(
        table,
        "A_age_by_base_code",
    )

    tests.append(result)
    tables["A_age_by_code"] = table
    expected_tables["A_age_by_code_expected"] = expected

    # ------------------------------------------------------------
    # 10. Баланс возраста B-кандидата по base_code
    # ------------------------------------------------------------
    table = pd.crosstab(df["base_code"], df["B_age"])

    result, expected = run_chi_square(
        table,
        "B_age_by_base_code",
    )

    tests.append(result)
    tables["B_age_by_code"] = table
    expected_tables["B_age_by_code_expected"] = expected

    # ------------------------------------------------------------
    # 11. Дополнительно: баланс пола по issue_block P/E/Nopos
    # ------------------------------------------------------------
    table = pd.crosstab(df["issue_block"], df["A_gender"])

    result, expected = run_chi_square(
        table,
        "A_gender_by_issue_block",
    )

    tests.append(result)
    tables["A_gender_by_issue"] = table
    expected_tables["A_gender_by_issue_expected"] = expected

    table = pd.crosstab(df["issue_block"], df["B_gender"])

    result, expected = run_chi_square(
        table,
        "B_gender_by_issue_block",
    )

    tests.append(result)
    tables["B_gender_by_issue"] = table
    expected_tables["B_gender_by_issue_expected"] = expected

    # ------------------------------------------------------------
    # 12. Дополнительно: баланс возраста по issue_block P/E/Nopos
    # ------------------------------------------------------------
    table = pd.crosstab(df["issue_block"], df["A_age"])

    result, expected = run_chi_square(
        table,
        "A_age_by_issue_block",
    )

    tests.append(result)
    tables["A_age_by_issue"] = table
    expected_tables["A_age_by_issue_expected"] = expected

    table = pd.crosstab(df["issue_block"], df["B_age"])

    result, expected = run_chi_square(
        table,
        "B_age_by_issue_block",
    )

    tests.append(result)
    tables["B_age_by_issue"] = table
    expected_tables["B_age_by_issue_expected"] = expected

    # ------------------------------------------------------------
    # Сводная таблица результатов
    # ------------------------------------------------------------
    results_df = pd.DataFrame(tests)

    results_df["significant_0_05"] = results_df["p_value"] < 0.05

    results_df["warning_low_expected_counts"] = (
        results_df["min_expected_count"] < 5
    )

    print("\nРезультаты χ²-тестов:")
    print(results_df[[
        "test",
        "chi2",
        "dof",
        "p_value",
        "min_expected_count",
        "share_expected_cells_lt_5",
        "significant_0_05",
        "warning_low_expected_counts",
    ]])

    # ------------------------------------------------------------
    # Сохраняем всё в Excel
    # ------------------------------------------------------------
    with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
        results_df.to_excel(writer, sheet_name="summary", index=False)

        for name, table in tables.items():
            table.to_excel(writer, sheet_name=safe_sheet_name(name))

        for name, table in expected_tables.items():
            table.to_excel(writer, sheet_name=safe_sheet_name(name))

    print(f"\nСохранено: {OUTPUT_FILE}")


if __name__ == "__main__":
    main()


Результаты χ²-тестов:
                          test       chi2  dof   p_value  min_expected_count  \
0   A_vs_B_gender_distribution   0.114309    1  0.735291           34.500000   
1      A_vs_B_age_distribution   1.193518    2  0.550593           22.000000   
2      A_gender_by_vignette_id   6.965602    8  0.540348            3.300000   
3      B_gender_by_vignette_id   6.162173    8  0.629072            3.400000   
4         A_age_by_vignette_id  15.725684   16  0.472265            2.000000   
5         B_age_by_vignette_id  15.858954   16  0.462847            2.100000   
6        A_gender_by_base_code   6.965602    8  0.540348            3.300000   
7        B_gender_by_base_code   6.162173    8  0.629072            3.400000   
8           A_age_by_base_code  15.725684   16  0.472265            2.000000   
9           B_age_by_base_code  15.858954   16  0.462847            2.100000   
10     A_gender_by_issue_block   5.121485    2  0.077247            4.714286   
11     B_gender_b

In [ ]:
df = pd.read_excel('/content/vignettes_10_sets_after_removal_with_codes_shuffled.xlsx')
var_name = []
for i in range (1,11):
  for l in (['A', 'B', 'C', 'D', 'E', 'F', 'G']):
    var_name.append(f'G{i}{l}')

In [ ]:
df['var_name'] = var_name

In [ ]:
df.to_excel('виньетки_итог.xlsx')